In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np


import os

In [43]:
#Programme d'aggrégation des fichiers meteo synop mensuel téléchargés à partir du site de météo-france

# remarques anciens fichiers synop à l'adresse https://www.data.gouv.fr/datasets/archive-synop-omm

In [3]:
#Répertoire des données brutes
rep="/home/dominique/DATA/meteo/données_climat/"

#Suffixe des fichiers des données brutes
suff1=".csv"
suff2=".json"



In [4]:
#Liste des fichiers de rep

contenu_rep=os.listdir(rep)

In [5]:
#Liste des fichiers par type
liste_csv=[]
liste_json=[]
liste_autre=[]

#Formation des listes
for fich in contenu_rep:
    suffixe=fich[-4:]   #formulation plus générale à trouver : couper au point 
    if suffixe==suff1:
        liste_csv.append(fich)
    elif suffixe=="json":
        liste_json.append(fich)
    else:
        liste_autre.append(fich)

#Effectif des listes
nb_csv=len(liste_csv)
nb_json=len(liste_json)
nb_autre=len(liste_autre)

#Affichage
print(f"Nous avons {nb_csv} fichiers de type csv , {nb_json} de type json et {nb_autre} d'autres types.")

Nous avons 360 fichiers de type csv , 1 de type json et 0 d'autres types.


In [6]:
#Exploration liste csv
liste_csv

['climat.200910.csv',
 'climat.200610.csv',
 'climat.202411.csv',
 'climat.200001.csv',
 'climat.200911.csv',
 'climat.199807.csv',
 'climat.202102.csv',
 'climat.201708.csv',
 'climat.202405.csv',
 'climat.201012.csv',
 'climat.202002.csv',
 'climat.201808.csv',
 'climat.200810.csv',
 'climat.201508.csv',
 'climat.199709.csv',
 'climat.199701.csv',
 'climat.200208.csv',
 'climat.202107.csv',
 'climat.199810.csv',
 'climat.200908.csv',
 'climat.201408.csv',
 'climat.202304.csv',
 'climat.199802.csv',
 'climat.200807.csv',
 'climat.201301.csv',
 'climat.200805.csv',
 'climat.199712.csv',
 'climat.201601.csv',
 'climat.201512.csv',
 'climat.200406.csv',
 'climat.202109.csv',
 'climat.201310.csv',
 'climat.200111.csv',
 'climat.199908.csv',
 'climat.202505.csv',
 'climat.199710.csv',
 'climat.201510.csv',
 'climat.201609.csv',
 'climat.200608.csv',
 'climat.199703.csv',
 'climat.202104.csv',
 'climat.201701.csv',
 'climat.202308.csv',
 'climat.201404.csv',
 'climat.199607.csv',
 'climat.2

In [7]:
#Lecture et concatenation des fichiers mois --> toutes variables

In [8]:
#intialisation avec premier fichier de la liste
df=pd.read_csv(rep+liste_csv[0],sep=";")

#incrementation
for i,fichier in enumerate(liste_csv):
    if i>0:
        df_fich=pd.read_csv(rep+fichier,sep=";")
        df=pd.concat([df_fich,df],axis=0)
#Réindexation
df=df.reset_index(drop=True)
#Dimension du fichier 
nb_lign=df.shape[0]
nb_col=df.shape[1]

print(f"Le fichier complet comporte {nb_lign} lignes et {nb_col} colonnes.")

Le fichier complet comporte 19355 lignes et 54 colonnes.


In [9]:
df.dtypes

NUM_POSTE        int64
DAT             object
PSTATM         float64
PMERM          float64
TMMOY          float64
TMSIGMA        float64
TXMOY          float64
TNMOY          float64
TSVMOY         float64
RR             float64
NBJRR1         float64
INST           float64
NBPMER         float64
NBTM           float64
NBTX           float64
NBTSV          float64
NBRR           float64
NBINS          float64
NBJTX25        float64
NBJTX30        float64
NBJTX35        float64
NBJTX40        float64
NBJTX0         float64
NBJTN0         float64
NBJRR5         float64
NBJRR10        float64
NBJRR50        float64
NBJRR100       float64
NBJNEIG0       float64
NBJNEIG1       float64
NBJNEIG10      float64
NBJNEIG50      float64
NBJFF10        float64
NBJFF20        float64
NBJFF30        float64
NBJVISI50      float64
NBJVISI100     float64
NBJVISI1000    float64
TMXAB          float64
TMXDAT          object
TMNAB          float64
TMNDAT          object
TXAB           float64
TXDAT      

In [10]:
liste_variables=list(df.columns)

In [11]:
#Analyse des données manquantes et % manquant par ligne et colonne

In [12]:
##Création d'un dataframe miroir codé en binaire : 0 donnée présente - 1 donnée absente
#Copie du df
df_mq=df
#transformation, remplacement des colonnes sauf la première colonne qui donne les n° de postes (=stations)
for i,var in enumerate(liste_variables[1:]):
    nom=var+"_mq"
    df_mq[nom]=df_mq[var].apply(lambda x:1 if pd.isna(x) else 0)
    df_mq=df_mq.drop([var],axis=1)

In [13]:
#Calcul des sommes par colonnes 
df_mq_col=pd.DataFrame(liste_variables,columns=["Variable"])
df_mq_col["Total"]=list(df_mq.sum())

df_mq_col=df_mq_col.drop(0)
df_mq_col=df_mq_col.reset_index(drop=True)   

df_mq_col["Taux%"]=df_mq_col["Total"]*100/nb_lign
df_mq_col["Taux%"]=df_mq_col["Taux%"].apply(lambda x:np.round(x,1))

In [14]:
df_mq_col.sort_values(by="Taux%",ascending=False)

,Variable,Total,Taux%
30,NBJNEIG50,19046,98.4
29,NBJNEIG10,19046,98.4
26,NBJRR100,18597,96.1
27,NBJNEIG0,17962,92.8
28,NBJNEIG1,17962,92.8
19,NBJTX35,17501,90.4
20,NBJTX40,17500,90.4
21,NBJTX0,13086,67.6
22,NBJTN0,13090,67.6
34,NBJVISI50,10711,55.3


In [15]:
#Calcul des sommes par lignes
df_mq_1=df[["NUM_POSTE","DAT"]]
df_mq_2=df_mq.drop(["NUM_POSTE","DAT_mq"],axis=1)

#Réindexation
df_mq_1=df_mq_1.reset_index(drop=True)
df_mq_2=df_mq_2.reset_index(drop=True)

#On commence le total à la colonne 2  de df_mq_2
df_mq_2["Total"]=df_mq_2.sum(axis=1)
df_mq_2=df_mq_2[["Total"]]

#On concatène les 2
df_mq_lign=pd.concat([df_mq_1,df_mq_2],axis=1)
df_mq_lign["Taux%"]=df_mq_lign["Total"]/(nb_col-2)*100
df_mq_lign["Taux%"]=df_mq_lign["Taux%"].apply(lambda x:np.round(x,1))

In [16]:
df_mq_lign

,NUM_POSTE,DAT,Total,Taux%
0,7005,2000-10-01 00:00:00,11,21.2
1,7015,2000-10-01 00:00:00,13,25.0
2,7020,2000-10-01 00:00:00,15,28.8
3,7027,2000-10-01 00:00:00,13,25.0
4,7037,2000-10-01 00:00:00,11,21.2
...,...,...,...,...
19350,81401,2009-10-01 00:00:00,20,38.5
19351,81405,2009-10-01 00:00:00,9,17.3
19352,81408,2009-10-01 00:00:00,13,25.0
19353,81415,2009-10-01 00:00:00,13,25.0


In [17]:
df_mq_lign["Taux%"].max()

96.2

In [18]:
#FIN CONCATENATION des fichiers

In [19]:
liste_index=df.index

In [20]:
#analyse de la variable NBJTX30
df2=df.loc[liste_index,["NUM_POSTE","DAT","NBJTX30","TXAB"]]

In [21]:
df2["mq"]=df2["NBJTX30"].apply(lambda x : 1 if pd.isna(x) else 0)

In [22]:
an=[]
mois=[]
for i,date in enumerate(list(df2["DAT"])):
    an_date=date[0:4]
    mois_date=date[5:7]
    an.append(an_date)
    mois.append(mois_date)

df2["an"]=an
df2["mois"]=mois

In [23]:
#REDUCTION du NOMBRE de MANQUANTS de NBJTX30

##On ajoute une variable de test : est ce que la température max du mois est >=30
##Si la réponse est non (=0=, alors nécessairement NBJTX30 doit être nul pour le mois correspondant 
##Dans ce cas si NBJTX30 est manquant , on le remplace par 0
df2["TX>=30"]=df2["TXAB"].apply(lambda x:1 if x>=30 else 0)

##Boucle de correction
##Correction : NBJTTX30 passe à 0 lorsque TX>=30 = 0 avec TXAB non manquant

for i in range(df2.shape[0]):
    test1=df2.loc[i,"mq"]
    test2=df2.loc[i,"TXAB"]
    
    if (test1==1)  & (not pd.isna(test2)):
        #test sur TX>=30
        test3=df2.loc[i,"TX>=30"]
        #Corrections si TXW>=30 est à 0
        if test3==0:
            df2.loc[i,"TXAB"]=0
            df2.loc[i,"mq"]=0
            df2.loc[i,"NBJTX30"]=0

In [24]:
df2

,NUM_POSTE,DAT,NBJTX30,TXAB,mq,an,mois,TX>=30
0,7005,2000-10-01 00:00:00,0.0,0.0,0,2000,10,0
1,7015,2000-10-01 00:00:00,0.0,0.0,0,2000,10,0
2,7020,2000-10-01 00:00:00,0.0,0.0,0,2000,10,0
3,7027,2000-10-01 00:00:00,0.0,0.0,0,2000,10,0
4,7037,2000-10-01 00:00:00,0.0,0.0,0,2000,10,0
...,...,...,...,...,...,...,...,...
19350,81401,2009-10-01 00:00:00,NaN,35.8,1,2009,10,1
19351,81405,2009-10-01 00:00:00,31.0,34.3,0,2009,10,1
19352,81408,2009-10-01 00:00:00,31.0,36.0,0,2009,10,1
19353,81415,2009-10-01 00:00:00,31.0,36.7,0,2009,10,1


In [26]:
#Le taux total est très correct
#On analyse la répartition des manquants par poste , année et mois sur la version corrigée de df2

In [27]:
#Premier point : y-a-t-il des stations plus touchées que d'autres

df2_poste=df2.groupby(by=["NUM_POSTE"],as_index=False).agg({"mq":"sum","TXAB":"count"})
df2_poste=df2_poste.rename({"TXAB":"nombre"},axis=1)
df2_poste["taux%"]=df2_poste["mq"]*100/df2_poste["nombre"]
df2_poste=df2_poste.sort_values(by=["taux%"],ascending=False)

In [28]:
df2_poste

,NUM_POSTE,mq,nombre,taux%
37,7661,70,290,24.137931
56,81415,55,282,19.503546
43,61970,42,222,18.918919
52,78925,49,281,17.437722
48,61997,36,229,15.720524
42,61968,36,230,15.652174
45,61976,29,263,11.026616
53,81401,25,248,10.080645
46,61980,20,277,7.220217
44,61972,17,249,6.827309


In [29]:
df2_an=df2.groupby(by=["an"],as_index=False).agg({"mq":"sum","TXAB":"count"})
df2_an=df2_an.rename({"TXAB":"nombre"},axis=1)
df2_an["taux%"]=df2_an["mq"]*100/df2_an["nombre"]
df2_an=df2_an.sort_values(by=["taux%"],ascending=False)

In [30]:
df2_an

,an,mq,nombre,taux%
2,1998,75,503,14.910537
0,1996,38,496,7.661290
1,1997,40,540,7.407407
16,2012,25,671,3.725782
28,2024,21,678,3.097345
14,2010,20,667,2.998501
10,2006,19,637,2.982732
4,2000,13,505,2.574257
29,2025,17,679,2.503682
27,2023,17,681,2.496329


In [31]:
df2_mois=df2.groupby(by=["mois"],as_index=False).agg({"mq":"sum","TXAB":"count"})
df2_mois=df2_mois.rename({"TXAB":"nombre"},axis=1)
df2_mois["taux%"]=df2_mois["mq"]*100/df2_an["nombre"]
df2_mois=df2_mois.sort_values(by=["taux%"],ascending=False)

In [32]:
df2_mois

,mois,mq,nombre,taux%
0,01,51,1542,10.282258
9,10,51,1575,9.222423
5,06,45,1571,8.754864
1,02,47,1561,8.703704
2,03,42,1574,8.349901
3,04,38,1584,7.692308
6,07,38,1585,6.846847
8,09,34,1574,5.872193
7,08,34,1582,5.743243
4,05,28,1569,5.544554


In [40]:
#Lecture des stations au format geojson
##téléchargé à partie de https://www.data.gouv.fr/datasets/liste-des-stations-en-open-data-du-reseau-meteorologique-infoclimat-static-et-meteo-france-synop

import geopandas as gpd

In [41]:
info_stations=gpd.read_file("/home/dominique/DATA/meteo/stations_xhr.json")

In [42]:
info_stations

,id,name,elevation,license,country,departement,last_activity,geometry
0,07005,Abbeville,69,"{'code': 0, 'license': 'Etalab Open License', ...",FR,80,2026-02-02T09:00:00Z,POINT (1.834 50.136)
1,07015,Lille-Lesquin,47,"{'code': 0, 'license': 'Etalab Open License', ...",FR,59,2026-02-02T09:00:00Z,POINT (3.0975 50.57)
2,07020,Cap de La Hague,6,"{'code': 0, 'license': 'Etalab Open License', ...",FR,50,2026-02-02T09:00:00Z,POINT (-1.93983 49.72517)
3,07027,Caen-Carpiquet,67,"{'code': 0, 'license': 'Etalab Open License', ...",FR,14,2026-02-02T09:00:00Z,POINT (-0.45617 49.18)
4,07037,Rouen-Boos,156,"{'code': 0, 'license': 'Etalab Open License', ...",FR,76,2026-02-02T09:00:00Z,POINT (1.17833 49.3895)
...,...,...,...,...,...,...,...,...
1211,STATIC0455,Chartrettes,80,"{'code': 1, 'license': 'CC BY', 'url': 'https:...",FR,77,2026-02-02T09:20:00Z,POINT (2.70083 48.48808)
1212,STATIC0456,Échiré,37,"{'code': 2, 'license': 'NON-COMMERCIAL ONLY: C...",FR,79,2026-02-02T09:20:00Z,POINT (-0.4407 46.3873)
1213,STATIC0457,Chevenoz,830,"{'code': 2, 'license': 'NON-COMMERCIAL ONLY: C...",FR,74,2026-02-02T09:20:00Z,POINT (6.64816 46.3371)
1214,STATIC0458,Moret-Loing-et-Orvanne,56,"{'code': 1, 'license': 'CC BY', 'url': 'https:...",FR,77,2026-02-02T09:20:00Z,POINT (2.80943 48.3613)


In [44]:
info_stations.explore()

In [48]:
#Croisement avec les id disponibles dans le fichier des températures

##Mise au format string et ajout d'un 0
df2["NUM_POSTE"]=df2["NUM_POSTE"].astype("str")
df2["NUM_POSTE"]=df2["NUM_POSTE"].apply(lambda x: "0"+x if len(x)==4 else x)

##liste
liste_id_stations=list(np.unique(df2["NUM_POSTE"]))



In [51]:
info_stations=info_stations[info_stations["id"].isin(liste_id_stations)]

In [52]:
info_stations

,id,name,elevation,license,country,departement,last_activity,geometry
0,07005,Abbeville,69,"{'code': 0, 'license': 'Etalab Open License', ...",FR,80,2026-02-02T09:00:00Z,POINT (1.834 50.136)
1,07015,Lille-Lesquin,47,"{'code': 0, 'license': 'Etalab Open License', ...",FR,59,2026-02-02T09:00:00Z,POINT (3.0975 50.57)
2,07020,Cap de La Hague,6,"{'code': 0, 'license': 'Etalab Open License', ...",FR,50,2026-02-02T09:00:00Z,POINT (-1.93983 49.72517)
3,07027,Caen-Carpiquet,67,"{'code': 0, 'license': 'Etalab Open License', ...",FR,14,2026-02-02T09:00:00Z,POINT (-0.45617 49.18)
4,07037,Rouen-Boos,156,"{'code': 0, 'license': 'Etalab Open License', ...",FR,76,2026-02-02T09:00:00Z,POINT (1.17833 49.3895)
6,07110,Brest-Guipavas,92,"{'code': 0, 'license': 'Etalab Open License', ...",FR,29,2026-02-02T09:00:00Z,POINT (-4.39117 48.45383)
7,07117,Ploumanac'h - Perros,55,"{'code': 0, 'license': 'Etalab Open License', ...",FR,22,2026-02-02T09:00:00Z,POINT (-3.47317 48.82583)
8,07130,Rennes-St Jacques,36,"{'code': 0, 'license': 'Etalab Open License', ...",FR,35,2026-02-02T09:00:00Z,POINT (-1.734 48.06883)
9,07139,Alençon - Valframbert,143,"{'code': 0, 'license': 'Etalab Open License', ...",FR,61,2026-02-02T09:00:00Z,POINT (0.11017 48.4455)
10,07149,Orly - Athis-Mons,86,"{'code': 0, 'license': 'Etalab Open License', ...",FR,91,2026-02-02T09:00:00Z,POINT (2.397 48.718)


In [53]:
info_stations.explore()

In [54]:
#Préparation d'un fichier mensuel par station et année + ajou du nom de la station
##regroupement
df2_annuel=df2.groupby(by=["an","NUM_POSTE"] , as_index=False).agg({"NBJTX30":"sum"})

##réduction du df des infos stations (on laisse tomber les coordonnées pour l'instant)
df_info_stations=info_stations[["id","name"]]

##fusion
df2_annuel=df2_annuel.merge(right=df_info_stations,left_on="NUM_POSTE",right_on="id",how="left")

In [56]:
#Nettoyage
df2_annuel=df2_annuel.dropna()
df2_annuel=df2_annuel.drop(["NUM_POSTE"],axis=1)

#Renommage des colonnes
df2_annuel=df2_annuel.rename({"an":"Année","id":"ID_station","name":"Station","NBJTX30":"Jours>=30°C"},axis=1)

#Réarrangement des colonnes
df2_annuel=df2_annuel.reindex(["Année","ID_station","Jours>=30°C","Station"],axis=1)

In [57]:
df2_annuel

,Année,ID_station,Jours>=30°C,Station
0,1996,07005,1.0,Abbeville
1,1996,07015,2.0,Lille-Lesquin
2,1996,07020,0.0,Cap de La Hague
3,1996,07027,4.0,Caen-Carpiquet
4,1996,07037,4.0,Rouen-Boos
...,...,...,...,...
1627,2025,07661,0.0,Saint-Mandrier-sur-Mer - Cap Cépet
1628,2025,07690,22.0,Nice - Côte d'Azur
1629,2025,07747,60.0,Perpignan - Rivesaltes
1630,2025,07761,52.0,Ajaccio - Campo dell'Oro


In [61]:
#Nombre final de stations
nb_stations=len(list(np.unique(df2_annuel["ID_station"])))

In [62]:
print(f"Le nombre de stations rettenues pour les statistiques est {nb_stations}")

Le nombre de stations rettenues pour les statistiques est 40


In [60]:
#export
df2_annuel.to_csv("/home/dominique/DATA/meteo/meteo_climat_omm_annuel.csv",index=False)